# Spark Structured Streaming Demo

## Introduction
This notebook demonstrates a minimal **Spark Structured Streaming** application. 
Structured Streaming is a scalable and fault-tolerant stream processing engine built on the Spark SQL engine. You can express your streaming computation the same way you would express a batch computation on static data.

### Objectives
1.  Initialize a `SparkSession` connected to a Spark Master.
2.  Read a stream of data from a generated source (`rate`).
3.  Perform a basic transformation on the data.
4.  Write the processed stream to an in-memory table for inspection.
5.  Periodically query the results.
6.  Perform aggregations using **Tumbling Windows**.
7.  Perform aggregations using **Sliding Windows**.

## Step 1: Initialize SparkSession

The `SparkSession` is the entry point to programming Spark with the Dataset and DataFrame API. 
Here, we configure it to connect to the Spark Master defined in our `docker-compose.yml` (service name `spark-master` on port `7077`).

In [1]:
import os
import sys

# --- Environment Setup for Spark 4.0.1 ---
# The container has a built-in Spark 3.5.0, but we installed 4.0.1 via pip.
# We need to point SPARK_HOME and sys.path to the new version.

NEW_SPARK_HOME = "/opt/conda/lib/python3.11/site-packages/pyspark"

if os.path.exists(NEW_SPARK_HOME):
    print(f"Found pip-installed Spark at: {NEW_SPARK_HOME}")
    
    # 1. Set SPARK_HOME to the new location
    os.environ['SPARK_HOME'] = NEW_SPARK_HOME
    
    # 2. Clean up sys.path to remove old Spark paths
    # The default image adds /usr/local/spark/python and py4j to sys.path
    paths_to_remove = [p for p in sys.path if "/usr/local/spark" in p]
    for p in paths_to_remove:
        sys.path.remove(p)
        
    # 3. Ensure the new location is importable (it should be in site-packages already)
else:
    print("Warning: Pip-installed Spark not found. Falling back to default.")

import pyspark
from pyspark.sql import SparkSession
from pyspark.context import SparkContext
from pyspark.sql.functions import col, window, avg

print(f"PySpark Version: {pyspark.__version__}")
print(f"SPARK_HOME: {os.environ.get('SPARK_HOME')}")

# Force stop any existing SparkContext
try:
    sc = SparkContext.getOrCreate()
    sc.stop()
except Exception:
    pass

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("StructuredStreamingDemo") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("SparkSession initialized successfully.")

Found pip-installed Spark at: /opt/conda/lib/python3.11/site-packages/pyspark
PySpark Version: 4.0.1
SPARK_HOME: /opt/conda/lib/python3.11/site-packages/pyspark
SparkSession initialized successfully.


## Step 2: Create a Streaming DataFrame

We use the `rate` source, which is a testing source provided by Spark. It generates data at the specified number of rows per second. Each generated row contains a `timestamp` and a `value` (auto-incrementing long).

In [2]:
# Create a streaming DataFrame using the rate source
# rowsPerSecond: How many rows to generate per second
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

print("Streaming DataFrame created. Schema:")
rate_df.printSchema()

Streaming DataFrame created. Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



## Step 3: Apply Transformations

Structured Streaming allows you to apply standard DataFrame operations. Here, we simply add a new column `value_plus_10` which adds 10 to the generated `value`.

In [3]:
# Perform a simple transformation: add 10 to the value
transformed_df = rate_df.withColumn("value_plus_10", col("value") + 10)

print("Transformation defined.")

Transformation defined.


## Step 4: Start the Streaming Query

We define the output sink and start the query. 
- **Output Mode `append`**: Only the new rows added to the Result Table since the last trigger will be outputted to the sink.
- **Format `memory`**: The output is stored in an in-memory table, which we can query using SQL. **Note**: This is for debugging purposes only and should not be used in production as it can cause OOM errors.
- **Query Name**: The name of the in-memory table (`rate_table`).

In [4]:
# Start the streaming query, writing to the memory sink
query = transformed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("rate_table") \
    .start()

print("Streaming query started...")

Streaming query started...


## Step 5: Monitor the Results

Since the data is being written to the `rate_table` in memory, we can use standard Spark SQL to query it. We will run a loop to query the table every 2 seconds to see the data arriving.

In [5]:
import time

# Query the memory table periodically to show results
print("Querying the in-memory table for 10 seconds...")
try:
    for i in range(5):
        print(f"\n--- Query {i+1} ---")
        spark.sql("SELECT * FROM rate_table ORDER BY timestamp DESC LIMIT 5").show()
        time.sleep(2)
except KeyboardInterrupt:
    print("Monitoring stopped by user.")

Querying the in-memory table for 10 seconds...

--- Query 1 ---
+---------+-----+-------------+
|timestamp|value|value_plus_10|
+---------+-----+-------------+
+---------+-----+-------------+


--- Query 2 ---
+--------------------+-----+-------------+
|           timestamp|value|value_plus_10|
+--------------------+-----+-------------+
|2025-11-26 16:34:...|    1|           11|
|2025-11-26 16:34:...|    0|           10|
+--------------------+-----+-------------+


--- Query 3 ---
+--------------------+-----+-------------+
|           timestamp|value|value_plus_10|
+--------------------+-----+-------------+
|2025-11-26 16:34:...|    3|           13|
|2025-11-26 16:34:...|    2|           12|
|2025-11-26 16:34:...|    1|           11|
|2025-11-26 16:34:...|    0|           10|
+--------------------+-----+-------------+


--- Query 4 ---
+--------------------+-----+-------------+
|           timestamp|value|value_plus_10|
+--------------------+-----+-------------+
|2025-11-26 16:34:...| 

## Step 6: Stop the Query

Always remember to stop the streaming query when you are done to release resources.

In [6]:
# Stop the query
if 'query' in locals() and query.isActive:
    query.stop()
    print("Query stopped.")
else:
    print("No active query to stop.")

Query stopped.


## Step 7: Tumbling Window Aggregation

**Tumbling windows** are fixed-sized, non-overlapping and contiguous time intervals. 
Here, we compute the average value for every **10-second** window.

In [7]:
# Tumbling Window: 10 seconds
tumbling_window_df = rate_df \
    .groupBy(window(col("timestamp"), "10 seconds")) \
    .agg(avg("value").alias("avg_value")) \
    .orderBy("window")

print("Tumbling window aggregation defined.")
tumbling_window_df.printSchema()

Tumbling window aggregation defined.
root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- avg_value: double (nullable = true)



In [8]:
# Start the tumbling window query
tumbling_query = tumbling_window_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("tumbling_table") \
    .start()

print("Tumbling window query started...")

Tumbling window query started...


In [9]:
print("Monitoring tumbling windows for 15 seconds...")
try:
    for i in range(5):
        print(f"\n--- Tumbling Query {i+1} ---")
        spark.sql("SELECT * FROM tumbling_table ORDER BY window DESC").show(truncate=False)
        time.sleep(3)
except KeyboardInterrupt:
    print("Monitoring stopped.")

# Stop the tumbling query
if 'tumbling_query' in locals() and tumbling_query.isActive:
    tumbling_query.stop()
    print("Tumbling query stopped.")

Monitoring tumbling windows for 15 seconds...

--- Tumbling Query 1 ---
+------+---------+
|window|avg_value|
+------+---------+
+------+---------+


--- Tumbling Query 2 ---
+------------------------------------------+---------+
|window                                    |avg_value|
+------------------------------------------+---------+
|{2025-11-26 16:34:30, 2025-11-26 16:34:40}|0.5      |
+------------------------------------------+---------+


--- Tumbling Query 3 ---
+------------------------------------------+---------+
|window                                    |avg_value|
+------------------------------------------+---------+
|{2025-11-26 16:34:30, 2025-11-26 16:34:40}|2.0      |
+------------------------------------------+---------+


--- Tumbling Query 4 ---
+------------------------------------------+---------+
|window                                    |avg_value|
+------------------------------------------+---------+
|{2025-11-26 16:34:40, 2025-11-26 16:34:50}|6.5      |
|

## Step 8: Sliding Window Aggregation

**Sliding windows** are fixed-sized windows that can overlap. They are defined by a window duration and a slide duration.
Here, we compute the average value for a **10-second window**, sliding every **5 seconds**.

In [10]:
# Sliding Window: 10 seconds duration, 5 seconds slide
sliding_window_df = rate_df \
    .groupBy(window(col("timestamp"), "10 seconds", "5 seconds")) \
    .agg(avg("value").alias("avg_value")) \
    .orderBy("window")

print("Sliding window aggregation defined.")
sliding_window_df.printSchema()

Sliding window aggregation defined.
root
 |-- window: struct (nullable = true)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- avg_value: double (nullable = true)



In [11]:
# Start the sliding window query
sliding_query = sliding_window_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("sliding_table") \
    .start()

print("Sliding window query started...")

Sliding window query started...


In [12]:
print("Monitoring sliding windows for 15 seconds...")
try:
    for i in range(5):
        print(f"\n--- Sliding Query {i+1} ---")
        spark.sql("SELECT * FROM sliding_table ORDER BY window DESC").show(truncate=False)
        time.sleep(3)
except KeyboardInterrupt:
    print("Monitoring stopped.")

# Stop the sliding query
if 'sliding_query' in locals() and sliding_query.isActive:
    sliding_query.stop()
    print("Sliding query stopped.")

Monitoring sliding windows for 15 seconds...

--- Sliding Query 1 ---
+------+---------+
|window|avg_value|
+------+---------+
+------+---------+


--- Sliding Query 2 ---
+------------------------------------------+---------+
|window                                    |avg_value|
+------------------------------------------+---------+
|{2025-11-26 16:34:50, 2025-11-26 16:35:00}|0.5      |
|{2025-11-26 16:34:45, 2025-11-26 16:34:55}|0.5      |
+------------------------------------------+---------+


--- Sliding Query 3 ---
+------------------------------------------+---------+
|window                                    |avg_value|
+------------------------------------------+---------+
|{2025-11-26 16:34:50, 2025-11-26 16:35:00}|2.0      |
|{2025-11-26 16:34:45, 2025-11-26 16:34:55}|2.0      |
+------------------------------------------+---------+


--- Sliding Query 4 ---
+------------------------------------------+---------+
|window                                    |avg_value|
+-----